In [1]:
# Skipped in CI: Colab/bootstrap dependency install cell.


# SegmentTable: 基本

:::{tip}
可視化、GravitySpyとの連携、高度な操作を含むより包括的なガイドについては、[テーブル / セグメント ユーザーガイド](intro_table.ipynb)を参照してください。
:::

# SegmentTable の基本

このチュートリアルでは、**GWexpy** におけるセグメント（時間区間）ベースの解析コンテナである `SegmentTable` を紹介します。通常の `EventTable` とは異なり、`SegmentTable` は区間のメタデータだけでなく、対応するペイロード（TimeSeries や ASD など）を遅延ロード（Lazy loading）形式で保持することに特化しています。

## GWpy 基盤クラスと gwexpy 拡張

セグメント解析の土台は GWpy にあります。各行の区間は `gwpy.segments.Segment` をそのまま使い、ペイロード列には GWpy の `TimeSeries` や `FrequencySeries` をそのまま載せられます。

そのうえで gwexpy は、`gwpy.table.Table` を拡張した `SegmentTable` と、遅延ロード可能な `SegmentCell` を提供し、`apply()` / `map()` / `crop()` / `asd()` のような表ベースの一括処理を追加しています。つまり、GWpy の基本型を保ったまま、複数セグメントをまとめて処理するワークフローを gwexpy が補強する構成です。


In [2]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import warnings

with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    import numpy as np
    from gwpy.segments import Segment

    from gwexpy.table import SegmentTable

    # 1. Create simple segments
    segs = [Segment(0, 4), Segment(4, 8), Segment(8, 12)]
    st = SegmentTable.from_segments(segs, label=["A", "B", "C"])
    st


## SegmentCell による遅延ロード

必要になるまでデータをロードしない「ペイロード列」を追加できます。これは巨大なデータのバッチ処理に非常に有効です。

In [3]:
def my_loader():
    # Simulate loading data
    print("Loading series...")
    from gwpy.timeseries import TimeSeries
    return TimeSeries(np.random.randn(128), sample_rate=32)

# Add a payload column with a loader (sequence of callables)
st.add_series_column("raw", loader=[my_loader]*len(st), kind="timeseries")
st

,span,label,raw
0,"(0, 4)",A,<lazy: timeseries>
1,"(4, 8)",B,<lazy: timeseries>
2,"(8, 12)",C,<lazy: timeseries>


## 行単位の処理

`SegmentTable` は `apply()` メソッドを提供し、各行を処理して新しい列として統合できます。

In [4]:
def process_row(row):
    span = row["span"]
    return {"duration": float(span[1] - span[0]), "valid": True}

st2 = st.apply(process_row)
st2.display()

,span,label,duration,valid,raw
0,"(0, 4)",A,4.0,True,<lazy: timeseries>
1,"(4, 8)",B,4.0,True,<lazy: timeseries>
2,"(8, 12)",C,4.0,True,<lazy: timeseries>


## 明示的ロードとデータ変換

`fetch()` や `materialize()` を使ってデータを明示的にロードできます。`to_pandas()` を使うと、通常の pandas DataFrame として扱えます。

In [5]:
st2.fetch()
df = st2.to_pandas()
df.head()

Loading series...
Loading series...
Loading series...


,span,label,duration,valid
0,"(0, 4)",A,4.0,True
1,"(4, 8)",B,4.0,True
2,"(8, 12)",C,4.0,True
